# Demo - Orchestration: failure, retry, alert and recovery

**Why we are doing this.** Kenny's complaint: managing dependent workflows, failures, retries, and notifications for failed jobs or missed service level agreements is hard and largely hand-built.

**What we will prove.**
1. A chain of dependent tasks runs in order: validate, then transform, then publish.
2. When a step fails, Snowflake retries it automatically.
3. When it still fails, a final step (the finalizer) emails the team, and nothing downstream runs with bad data.
4. After the fix, the chain resumes from the failed step. Steps that already succeeded are not rerun.
5. A separate alert emails when Gold has not been published on time.

**For the Databricks engineers:** this is the equivalent of a Databricks Workflows job with task dependencies, retries and a notification step, defined in SQL.

**How to read this notebook.** Every cell has one of four labels:
- **SETUP - ALREADY DONE (read only):** code that already exists in the account. It is shown so the engineers can see how it was built. You do not run it.
- **RUN DURING DEMO:** a SQL cell you run live. Click inside it and press the run button (or Cmd+Enter).
- **UI STEP:** something you do in the Snowsight screens or the AWS console, not in the notebook.
- **SAY THIS:** your talk track for that moment.

### SETUP - ALREADY DONE (read only, do not run)
**The gated Gold table**

```sql
-- Published claims for reports and clients. TARGET_LAG = 'DOWNSTREAM' and nothing reads from it,
-- so Snowflake never refreshes it on its own. Only the pipeline's transform step refreshes it,
-- and that step runs only after validation passes. This is the gate.
CREATE DYNAMIC TABLE SOMATUS_DEMO.GOLD.CLAIMS_PUBLISHED
  TARGET_LAG = 'DOWNSTREAM' WAREHOUSE = WH_TRANSFORM REFRESH_MODE = INCREMENTAL AS
SELECT CLAIM_ID, MEMBER_ID, CLIENT_ID, SERVICE_DATE, CPT_CODE, PAID_AMOUNT, CLAIM_STATUS, FILE_NAME, LOADED_AT
FROM SOMATUS_DEMO.SILVER.CLIENT_CLAIMS;
```

### SETUP - ALREADY DONE (read only, do not run)
**The task graph**

```sql
-- Root: starts the run. Retry the whole graph once automatically on failure.
CREATE TASK OPS.T_ROOT WAREHOUSE=WH_INGEST TASK_AUTO_RETRY_ATTEMPTS = 1
  AS CALL OPS.SP_START_RUN();
-- Children run only after their parent succeeds
CREATE TASK OPS.T_VALIDATE  WAREHOUSE=WH_INGEST    AFTER OPS.T_ROOT     AS CALL OPS.SP_VALIDATE_BATCH();
CREATE TASK OPS.T_TRANSFORM WAREHOUSE=WH_TRANSFORM AFTER OPS.T_VALIDATE
  AS ALTER DYNAMIC TABLE GOLD.CLAIMS_PUBLISHED REFRESH;   -- publish only after validation passes
CREATE TASK OPS.T_PUBLISH   WAREHOUSE=WH_TRANSFORM AFTER OPS.T_TRANSFORM
  AS INSERT INTO OPS.PIPELINE_LOG (STEP, STATUS, DETAIL) SELECT 'PUBLISH','OK','CLAIMS_PUBLISHED rows=' || COUNT(*) FROM GOLD.CLAIMS_PUBLISHED;
-- Finalizer: always runs at the end, success or failure, and emails the outcome
CREATE TASK OPS.T_NOTIFY WAREHOUSE=WH_INGEST FINALIZE = OPS.T_ROOT AS CALL OPS.SP_NOTIFY_RUN();

-- Production schedule (not used in the demo):
-- ALTER TASK T_ROOT SET SCHEDULE = 'USING CRON 0 5 * * * America/New_York';
```

### SETUP - ALREADY DONE (read only, do not run)
**The SLA alert**

```sql
CREATE ALERT OPS.ALERT_GOLD_SLA WAREHOUSE=WH_INGEST SCHEDULE='60 MINUTE'
IF (EXISTS (SELECT 1 FROM (SELECT MAX(LOGGED_AT) LAST_PUBLISH FROM OPS.PIPELINE_LOG WHERE STEP='PUBLISH' AND STATUS='OK')
            WHERE LAST_PUBLISH IS NULL OR LAST_PUBLISH < DATEADD('minute', -5, CURRENT_TIMESTAMP())))
THEN CALL SYSTEM$SEND_EMAIL('SOMATUS_DEMO_EMAIL', 'mayank.p.gupta@snowflake.com',
       '[Somatus demo] SLA MISSED - Gold claims not published', '...');
```

**How we make it fail on purpose.** The validation procedure reads a switch in the table OPS.DEMO_CONTROL. When FAIL_VALIDATE is 'TRUE', validation raises an error. This stands in for a real problem such as a client file with a wrong total.

### RUN DURING DEMO
Set the role and schema. Run this first, every time you open the notebook.

In [ ]:
%%sql -r ctx
USE ROLE ACCOUNTADMIN;
USE DATABASE SOMATUS_DEMO;
USE SCHEMA OPS;

### UI STEP - do this in the screen, not in the notebook
**Show the graph visually**

1. In the left menu, click **Data**, then **Databases**.
2. Open **SOMATUS_DEMO > OPS > Tasks** and click **T_ROOT**.
3. Click the **Graph** tab. You see T_ROOT, then T_VALIDATE, then T_TRANSFORM, then T_PUBLISH, with T_NOTIFY as the finalizer.
4. Keep this tab open. You will come back to it in Step 2 to see the failed step in red.

## Step 1 - A new batch of claims arrives

### RUN DURING DEMO
Insert 500 new claims into Bronze. They are now waiting for the pipeline to validate and publish them.

In [ ]:
%%sql -r new_batch
INSERT INTO SOMATUS_DEMO.BRONZE.CLAIMS_RAW
  (CLAIM_ID, MEMBER_ID, CLIENT_ID, SERVICE_DATE, CPT_CODE, PAID_AMOUNT, CLAIM_STATUS, FILE_NAME, LOADED_AT)
SELECT 'LIVE_' || UUID_STRING(), 'M00000001', 'HUMANA_DEMO', '2026-10-21', '90935', 450, 'PAID',
       'live_batch', CURRENT_TIMESTAMP()
FROM TABLE(GENERATOR(ROWCOUNT => 500));

### RUN DURING DEMO
You should see ARRIVED_IN_BRONZE = 500 and PUBLISHED_TO_GOLD = 0.

In [ ]:
%%sql -r pending
SELECT (SELECT COUNT(*) FROM SOMATUS_DEMO.BRONZE.CLAIMS_RAW WHERE FILE_NAME = 'live_batch') AS ARRIVED_IN_BRONZE,
       (SELECT COUNT(*) FROM SOMATUS_DEMO.GOLD.CLAIMS_PUBLISHED WHERE FILE_NAME = 'live_batch') AS PUBLISHED_TO_GOLD;

## Step 2 - Break it and run the pipeline

### RUN DURING DEMO
Turn on the failure switch, clear the log, and start the pipeline.

In [ ]:
%%sql -r break
UPDATE DEMO_CONTROL SET VAL = 'TRUE' WHERE KEY = 'FAIL_VALIDATE';
TRUNCATE TABLE PIPELINE_LOG;
EXECUTE TASK T_ROOT;

### RUN DURING DEMO
Run this about 30 seconds later, and again after another 30 seconds. You will see T_VALIDATE **FAILED**, then attempt 2 (the automatic retry) also **FAILED**, and T_NOTIFY **SUCCEEDED**. T_TRANSFORM and T_PUBLISH never ran.

In [ ]:
%%sql -r history
SELECT NAME, STATE, ATTEMPT_NUMBER, LEFT(ERROR_MESSAGE, 100) AS ERROR, SCHEDULED_TIME
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(SCHEDULED_TIME_RANGE_START => DATEADD('minute', -10, CURRENT_TIMESTAMP())))
WHERE NAME IN ('T_ROOT','T_VALIDATE','T_TRANSFORM','T_PUBLISH','T_NOTIFY')
ORDER BY SCHEDULED_TIME, NAME;

### RUN DURING DEMO
The pipeline's own log shows where it stopped.

In [ ]:
%%sql -r log_failed
SELECT * FROM PIPELINE_LOG ORDER BY LOGGED_AT;

### UI STEP - do this in the screen, not in the notebook
**Show the failure in the screen and the email**

1. Go back to the **T_ROOT > Graph** tab and refresh. T_VALIDATE is shown in red. Click it to see the error message.
2. Click the **Run History** tab to see both attempts.
3. Open your email inbox (mayank.p.gupta@snowflake.com) and show the failure email sent by the finalizer.

> **SAY THIS:** The failure was caught, retried, and reported, and Gold was never touched with bad data.

## Step 3 - Fix it and resume from the failed step

### RUN DURING DEMO
Turn the failure switch off (the fix), then resume the last run. RETRY LAST restarts from T_VALIDATE. T_ROOT, which already succeeded, is not rerun.

In [ ]:
%%sql -r fix_retry
UPDATE DEMO_CONTROL SET VAL = 'FALSE' WHERE KEY = 'FAIL_VALIDATE';
EXECUTE TASK T_ROOT RETRY LAST;

### RUN DURING DEMO
Run this about 30 seconds later. The latest attempt succeeds through T_VALIDATE, T_TRANSFORM and T_PUBLISH. A success email arrives.

In [ ]:
%%sql -r history_after
SELECT NAME, STATE, ATTEMPT_NUMBER, LEFT(ERROR_MESSAGE, 100) AS ERROR, SCHEDULED_TIME
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(SCHEDULED_TIME_RANGE_START => DATEADD('minute', -15, CURRENT_TIMESTAMP())))
WHERE NAME IN ('T_ROOT','T_VALIDATE','T_TRANSFORM','T_PUBLISH','T_NOTIFY')
ORDER BY SCHEDULED_TIME, NAME;

### RUN DURING DEMO
The log now shows PUBLISH OK.

In [ ]:
%%sql -r log_ok
SELECT * FROM PIPELINE_LOG ORDER BY LOGGED_AT;

### RUN DURING DEMO
You should see 500 and 500. The claims were published once, only after validation passed, even though the first run failed and was retried.

In [ ]:
%%sql -r published
SELECT (SELECT COUNT(*) FROM SOMATUS_DEMO.BRONZE.CLAIMS_RAW WHERE FILE_NAME = 'live_batch') AS ARRIVED_IN_BRONZE,
       (SELECT COUNT(*) FROM SOMATUS_DEMO.GOLD.CLAIMS_PUBLISHED WHERE FILE_NAME = 'live_batch') AS PUBLISHED_TO_GOLD;

> **SAY THIS:** Recovery is one command, from the exact point of failure, with no duplicate processing.

## Step 4 (talk only) - The SLA alert

> **SAY THIS (do not run in the 45-minute cut):** Failures are one problem. The other is a pipeline that simply runs late. A separate alert checks every hour whether Gold was published on time, and emails the team if it was not. That is the setup cell at the top of this notebook.

### IF ASKED - run only if someone wants to see it
Run the SLA check now. It emails only if Gold was not published in the last 5 minutes (a short window for the demo). Right after Step 3 it will not fire; it fires about 5 minutes later.

In [ ]:
%%sql -r alert
EXECUTE ALERT ALERT_GOLD_SLA;

### IF ASKED
Show the alert's history. STATE is TRIGGERED when the email was sent, CONDITION_FALSE when everything was on time.

In [ ]:
%%sql -r alert_history
SELECT NAME, STATE, SCHEDULED_TIME, SQL_ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.ALERT_HISTORY(SCHEDULED_TIME_RANGE_START => DATEADD('hour', -1, CURRENT_TIMESTAMP())))
ORDER BY SCHEDULED_TIME DESC;